# 🌾 PIPELINE INSTANCE SEGMENTATION - BỆNH LÁ LÚA (KAGGLE)
**Mô hình:** YOLOv26m &nbsp;|&nbsp; **Phần cứng yêu cầu:** Kaggle GPU (Tesla T4 x2 hoặc x1)  
**Mục tiêu:** Cung cấp quy trình huấn luyện end-to-end cho mô hình phát hiện và khoanh vùng vết bệnh trên lá lúa.

---
### 📖 HƯỚNG DẪN CÁCH CHẠY DÀNH CHO TEAM:
1. **Bật GPU trên Kaggle:** Ở panel bên phải -> `Session options` -> `Accelerator` -> Chọn `GPU T4 x2` hoặc `P100`. (Bật Internet = ON).
2. **Thêm dữ liệu (Add Data):** Nhấn nút `Add Input` (Góc trên phải) để đưa dataset ảnh lúa đã tiền xử lý vào notebook.
3. **Tùy biến Cấu hình:** Tất cả tham số (Epochs, Batch size, Lr...) đều nằm chung trong **CELL 1**. Bạn chỉ cần thay đổi giá trị ở đó.
4. **Trình tự chạy:**
   - **Train một mô hình mới:** Chạy tuần tự **BƯỚC 1 ➡️ BƯỚC 2A ➡️ BƯỚC 3 ➡️ BƯỚC 4**.
   - **Train tiếp (Resume):** Chạy tuần tự **BƯỚC 1 ➡️ BƯỚC 2B ➡️ BƯỚC ...**

---
## BƯỚC 1: IMPORT THƯ VIỆN & CẤU HÌNH TẬP TRUNG (CONFIG)
> ⚠️ **Hành động của Team:** Chỉ cần điền và chỉnh sửa các thống số trong Lớp `PipelineConfig` ở Cell code ngay bên dưới.
> - Tránh sửa can thiệp vào các hàm bên dưới để đảm bảo tính ổn định của hệ thống.
> - **Lưu ý:** Biến `BASE_INPUT_DIR` phải trỏ đúng vào thư mục dataset (được sinh ra sau khi bạn ấn Add Data).

In [ ]:
# ============================================================
# 1.1 — CÀI ĐẶT THƯ VIỆN (chạy 1 lần duy nhất trên Kaggle)
# ============================================================
# ⚠️ Pin Pillow < 11.2 để tránh lỗi PIL._typing._Ink
!pip install "Pillow==10.4.0" -q
# Cài ultralytics và các thư viện phụ trợ (KHÔNG dùng -U để tránh nâng Pillow)
!pip install ultralytics requests opencv-python matplotlib -q


# ============================================================
# 1.2 — IMPORT THƯ VIỆN
# ============================================================
import os
import time
import yaml
import shutil
import requests
import torch                        # Kiểm tra CUDA / VRAM
import cv2
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from pathlib import Path
from dataclasses import dataclass, field
from typing import List
from PIL import Image
from ultralytics import YOLO
import ultralytics

# ============================================================
# 1.3 — CẤU HÌNH TẬP TRUNG (CONFIG / CONSTANTS)
# ============================================================
# Team chỉ cần thay đổi giá trị ở đây, logic bên dưới tự nhận.
# Dùng frozen dataclass để tránh vô tình ghi đè config giữa chừng.

@dataclass(frozen=True)
class PipelineConfig:
    """Tập trung toàn bộ siêu tham số & đường dẫn cho pipeline."""

    # ---------- ĐƯỜNG DẪN ----------
    BASE_INPUT_DIR: str = '/kaggle/input/datasets/phantunqucanh/preprocessed-dataset/PTIT_RICE_SEGMENTATION'       # ⚠️ SỬA ĐƯỜNG DẪN NÀY KHỚP VỚI DATASET BẠN ĐÃ THÊM VÀO
    WORKING_DIR: str = '/kaggle/working'         # Thư mục làm việc (output)
    PROJECT_DIR: str = '/kaggle/working/Rice_Disease_Seg'  # Lưu kết quả training
    RUN_NAME: str = 'yolo26m_seg_train'          # Tên run (tạo subfolder)

    # ---------- MÔ HÌNH ----------
    # Nếu 'yolo26m-seg' là file trọng số custom (.pt) đặt trên Kaggle:
    #   → đặt đường dẫn đầy đủ, ví dụ '/kaggle/input/weights/yolo26m-seg.pt'
    # Nếu là mã biến thể Ultralytics (vd: yolo11m-seg, yolo8m-seg):
    #   → ghi tên chuẩn Ultralytics, thư viện tự download.
    MODEL_WEIGHTS: str = 'yolo26m-seg.pt'

    # ---------- SIÊU THAM SỐ HUẤN LUYỆN ----------
    EPOCHS: int = 120                # Số epoch tối đa
    PATIENCE: int = 20               # Early Stopping: dừng nếu mAP không cải thiện sau N epoch
    BATCH_SIZE: int = 16             # Batch size an toàn cho T4-16GB ở imgsz=768
    IMAGE_SIZE: int = 768            # Kích thước ảnh train (768 giữ chi tiết vết bệnh nhỏ)
    WORKERS: int = 4                 # Số luồng DataLoader (Kaggle có 4 CPU cores)
    DEVICE: str = '0,1'                  # GPU index (0 = GPU đầu tiên)
    OPTIMIZER: str = 'AdamW'         # Thuật toán tối ưu
    LR0: float = 0.001              # Learning rate khởi đầu
    LRF: float = 0.01               # Learning rate cuối (tỷ lệ so với lr0)
    WEIGHT_DECAY: float = 0.0005     # L2 Regularization
    WARMUP_EPOCHS: int = 5           # Số epoch khởi động (warm-up)
    COS_LR: bool = True             # Sử dụng Cosine Annealing LR Scheduler
    AMP: bool = True                # Mixed Precision (FP16) — tiết kiệm ~30% VRAM

    # ---------- DATA AUGMENTATION (tối ưu cho ảnh thực vật) ----------
    MOSAIC: float = 1.0             # Trộn 4 ảnh → chống center-bias
    CLOSE_MOSAIC: int = 15          # Tắt mosaic N epoch cuối → ổn định hội tụ
    MIXUP: float = 0.15             # Trộn 2 ảnh mờ → tăng tính tổng quát
    COPY_PASTE: float = 0.3         # Copy-paste object → tăng recall cho vết bệnh nhỏ
    DEGREES: float = 15.0           # Xoay ngẫu nhiên (lá lúa có nhiều hướng)
    TRANSLATE: float = 0.1          # Dịch chuyển ảnh 10%
    SCALE: float = 0.5              # Phóng/thu ± 50%
    SHEAR: float = 2.0              # Biến dạng nhẹ
    FLIPUD: float = 0.5             # Lật dọc 50% (lá có thể úp/ngửa)
    FLIPLR: float = 0.5             # Lật ngang 50%
    HSV_H: float = 0.02            # Biến đổi Hue ± 2% (nhạy với màu lá)
    HSV_S: float = 0.7             # Biến đổi Saturation ± 70% (mô phỏng ánh sáng)
    HSV_V: float = 0.4             # Biến đổi Value ± 40% (mô phỏng bóng râm)
    ERASING: float = 0.3           # Random Erasing → chống overfitting vùng cụ thể

    # ---------- INFERENCE ----------
    CONF_THRESH: float = 0.25       # Ngưỡng confidence cho prediction
    IOU_THRESH: float = 0.45        # Ngưỡng IoU cho NMS
    INFER_IMG_SIZE: int = 1024      # Kích thước ảnh inference (to hơn → chi tiết hơn)
    AGNOSTIC_NMS: bool = True       # NMS bỏ qua class → tránh box đè nhau

    # ---------- DATASET ----------
    NUM_CLASSES: int = 7
    CLASS_NAMES: tuple = (
        'Bacterial blight',         # Bạc lá
        'Brown spot',               # Đốm nâu
        'Grain discoloration',      # Biến màu hạt
        'Leaf blast',               # Đạo ôn lá
        'Leaf scald',               # Cháy lá
        'Narrow brown',             # Đốm sọc nâu
        'Pesticide residue',        # Dư lượng thuốc
    )

    # ---------- URL ẢNH THỰC TẾ (PRACTICAL) ----------
    # URL ảnh ngoài thực tế — KHÔNG nằm trong tập train/valid/test
    # Dùng để kiểm chứng model trên dữ liệu "chưa bao giờ thấy" (practical)
    SAMPLE_PRACTICAL_URL: str = (
        '/kaggle/input/datasets/phantunqucanh/pratical-set/dao_on_2.jpg'
    )

    # ---------- PROPERTY TIỆN ÍCH ----------
    @property
    def yaml_path(self) -> str:
        """Đường dẫn file data.yaml được sinh ra."""
        return os.path.join(self.WORKING_DIR, 'data.yaml')

    @property
    def best_weights_path(self) -> str:
        """Đường dẫn tới file best.pt sau khi train xong."""
        return os.path.join(self.PROJECT_DIR, self.RUN_NAME, 'weights', 'best.pt')

    @property
    def last_weights_path(self) -> str:
        """Đường dẫn tới file last.pt (checkpoint cuối cùng)."""
        return os.path.join(self.PROJECT_DIR, self.RUN_NAME, 'weights', 'last.pt')


# Khởi tạo config duy nhất cho toàn bộ pipeline
CFG = PipelineConfig()

# ============================================================
# 1.4 — KIỂM TRA MÔI TRƯỜNG (ĐÃ FIX LỖI MULTI-GPU)
# ============================================================
print('=' * 60)
print('⚙️  KIỂM TRA CẤU HÌNH HỆ THỐNG')
print('=' * 60)
ultralytics.checks()  # In thông tin GPU, CUDA, thư viện

if torch.cuda.is_available():
    # Xử lý chuỗi '0,1' thành số 0 để hàm PyTorch đọc được
    first_device = str(CFG.DEVICE).split(',')[0].strip()
    device_id = int(first_device) if first_device.isdigit() else 0
    
    gpu_name = torch.cuda.get_device_name(device_id)
    vram_gb = torch.cuda.get_device_properties(device_id).total_memory / (1024**3)
    
    # Nếu dùng multi-gpu thì nhân đôi hiển thị VRAM cho chuẩn
    num_gpus = len(str(CFG.DEVICE).split(',')) if isinstance(CFG.DEVICE, str) else 1
    total_vram = vram_gb * num_gpus
    
    print(f'🖥️  GPU: {num_gpus}x {gpu_name} | Tổng VRAM: {total_vram:.1f} GB')
    print(f'🔧  AMP (FP16): {"BẬT" if CFG.AMP else "TẮT"}')
    print(f'📐  Image size: {CFG.IMAGE_SIZE} | Batch: {CFG.BATCH_SIZE}')
else:
    print('⚠️  KHÔNG PHÁT HIỆN GPU – training sẽ rất chậm!')

print(f'📁  Model weights: {CFG.MODEL_WEIGHTS}')


---
## BƯỚC 2A: CHUẨN BỊ DỮ LIỆU & HUẤN LUYỆN (TRAINING TỪ ĐẦU)
> 🚀 **Hành động của Team:** Chạy Cell này nếu bạn **bắt đầu huấn luyện một mô hình mới**.
> - Đoạn mã sẽ tự động quét thư mục `train` và `valid`, tạo file cấu hình `data.yaml` cho YOLO và bắt đầu vòng lặp Train.
> - Tiến trình sẽ in liên tục bên dưới để bạn theo dõi. (Quá trình này tốn từ vài chục phút tới vài giờ).

In [ ]:
# ============================================================
# 2.1 — TÌM KIẾM DATASET TỰ ĐỘNG
# ============================================================
def find_dataset(base_dir: str) -> str:
    print('\n🔍 Đang quét tìm dataset...')
    for root, dirs, _ in os.walk(base_dir):
        if 'train' in dirs and 'valid' in dirs:
            train_images = os.path.join(root, 'train', 'images')
            if os.path.isdir(train_images):
                print(f'✅ Dataset tìm thấy tại: {root}')
                return root
    raise FileNotFoundError(
        '❌ Không tìm thấy thư mục dataset hợp lệ (cần train/images & valid/images).\n'
        f'   Đã quét trong: {base_dir}'
    )

# ============================================================
# 2.2 — TẠO FILE data.yaml CHO ULTRALYTICS
# ============================================================
def create_data_yaml(dataset_path: str, cfg) -> str:
    data_config = {
        'train': os.path.join(dataset_path, 'train', 'images'),
        'val': os.path.join(dataset_path, 'valid', 'images'),
        'nc': cfg.NUM_CLASSES,
        'names': list(cfg.CLASS_NAMES),
    }

    test_images = os.path.join(dataset_path, 'test', 'images')
    if os.path.isdir(test_images):
        data_config['test'] = test_images
        print(f'📂 Tìm thấy tập TEST: {test_images}')
    else:
        print('ℹ️  Không tìm thấy thư mục test/images — bỏ qua tập test')

    os.makedirs(cfg.WORKING_DIR, exist_ok=True)
    with open(cfg.yaml_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_config, f, sort_keys=False, allow_unicode=True)

    print(f'✅ Đã tạo data.yaml tại: {cfg.yaml_path}')
    return cfg.yaml_path

# ============================================================
# 2.3 — KHỞI TẠO MÔ HÌNH
# ============================================================
def build_model(cfg) -> YOLO:
    weight_path = cfg.MODEL_WEIGHTS
    if os.path.isabs(weight_path) and not os.path.exists(weight_path):
        raise FileNotFoundError(f'❌ Không tìm thấy file trọng số: {weight_path}')

    print(f'\n🚀 Đang load mô hình: {weight_path}')
    model = YOLO(weight_path)
    print(f'✅ Mô hình đã sẵn sàng — Architecture: {model.model.__class__.__name__}')
    return model

# ============================================================
# 2.4 — HÀM HUẤN LUYỆN CHÍNH (ĐÃ FIX CUDA ERROR)
# ============================================================
def run_training(model: YOLO, yaml_path: str, cfg):
    print('\n' + '=' * 60)
    print('⏳ BẮT ĐẦU HUẤN LUYỆN')
    print(f'   Epochs: {cfg.EPOCHS} | Batch : {cfg.BATCH_SIZE} | ImgSz: {cfg.IMAGE_SIZE}')
    print('=' * 60)

    # ĐẢM BẢO VRAM TRỐNG TRƯỚC KHI TRAIN ĐỂ TRÁNH LỖI "CUDA UNKNOWN ERROR"
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("🧹 Đã dọn dẹp cache GPU (VRAM).")

    start_time = time.time()

    results = model.train(
        data=yaml_path,
        epochs=cfg.EPOCHS,
        imgsz=cfg.IMAGE_SIZE,
        batch=cfg.BATCH_SIZE,

        workers=cfg.WORKERS,          
        device=cfg.DEVICE,            
        amp=cfg.AMP,                  

        patience=cfg.PATIENCE,        
        weight_decay=cfg.WEIGHT_DECAY,  
        optimizer=cfg.OPTIMIZER,
        lr0=cfg.LR0,                  
        lrf=cfg.LRF,                  
        warmup_epochs=cfg.WARMUP_EPOCHS,  
        cos_lr=cfg.COS_LR,            

        mosaic=cfg.MOSAIC,            
        close_mosaic=cfg.CLOSE_MOSAIC,  
        mixup=cfg.MIXUP,              
        copy_paste=cfg.COPY_PASTE,    
        degrees=cfg.DEGREES,          
        translate=cfg.TRANSLATE,      
        scale=cfg.SCALE,              
        shear=cfg.SHEAR,              
        flipud=cfg.FLIPUD,            
        fliplr=cfg.FLIPLR,            
        hsv_h=cfg.HSV_H,             
        hsv_s=cfg.HSV_S,             
        hsv_v=cfg.HSV_V,             
        erasing=cfg.ERASING,         

        project=cfg.PROJECT_DIR,
        name=cfg.RUN_NAME,
        exist_ok=True,               
        save=True,                   
        save_period=10,              
        plots=True,                  
        verbose=True,
    )

    elapsed = time.time() - start_time
    hours, remainder = divmod(elapsed, 3600)
    minutes, seconds = divmod(remainder, 60)

    print('\n' + '=' * 60)
    print(f'✅ HOÀN TẤT HUẤN LUYỆN trong {int(hours)}h {int(minutes)}m {int(seconds)}s')
    print(f'📂 Kết quả lưu tại: {cfg.PROJECT_DIR}/{cfg.RUN_NAME}')
    print('=' * 60)

    return results

# ============================================================
# 2.5 — CHẠY PIPELINE SETUP + TRAIN
# ============================================================
dataset_path = find_dataset(CFG.BASE_INPUT_DIR)
yaml_config_path = create_data_yaml(dataset_path, CFG)
model = build_model(CFG)
train_results = run_training(model, yaml_config_path, CFG)

---
## BƯỚC 2B: TIẾP TỤC HUẤN LUYỆN TỪ CHECKPOINT CŨ (RESUME)
> 🚨 **CẢNH BÁO:** Nếu bạn ĐÃ CHẠY **BƯỚC 2A**, thì nhớ **KHÔNG CHẠY** cell này nữa.
> - **Mục đích:** Chỉ chạy BƯỚC 2B nếu trước đó đang học thì bị ngắt quãng (vd hết giờ, rớt mạng Kaggle).
> - **Thao tác:** Đưa checkpoint cũ (VD: file `last.pt`) vào Kaggle thông qua Add Data và cập nhật đường dẫn `checkpoint_path` bên dưới.

In [ ]:
!pip install ultralytics requests opencv-python matplotlib -q

from ultralytics import YOLO

# Load lại chính file last.pt của ngày hôm qua
checkpoint_path = '/kaggle/input/đường_dẫn_của_bạn_đến_model_cũ/last.pt'  # ⚠️ SỬA ĐƯỜNG DẪN TỚI FILE LAST.PT NẾU MUỐN RESUME
model = YOLO(checkpoint_path)

print("🚀 Đang khôi phục quá trình học từ checkpoint...")

# CHỈ CẦN THÊM resume=True, YOLO sẽ tự động móc nối toàn bộ cấu hình cũ và chạy nốt đến 120


---
## BƯỚC 3: ĐÁNH GIÁ (TEST) VÀ DỰ ĐOÁN THỬ (INFERENCE)
> 📊 **Hành động của Team:** Sau khi hệ thống Train xong hoặc resume thành công, chạy Cell này để:
> 1. Gọi hàm tính độ chính xác trung bình (mAP) nghiêm ngặt trên tập **test** (dữ liệu mô hình hoàn toàn chưa nhìn thấy).
> 2. Gọi hàm dự đoán trên **1 tấm hình thực tế** để kiểm tra thị giác kết quả khoanh vùng (bounding box + mask).

In [ ]:
# ============================================================
# 3.1 — HÀM ĐÁNH GIÁ TRÊN TẬP TEST
# ============================================================
def evaluate_on_test(weights_path: str, cfg: PipelineConfig) -> None:
    """
    Đánh giá (validate) mô hình trên tập TEST — dữ liệu chưa từng
    thấy trong quá trình huấn luyện (khác với valid dùng để Early Stopping).

    In ra các chỉ số chính:
      - mAP50 (Segmentation): Độ chính xác trung bình tại IoU=0.5
      - mAP50-95 (Segmentation): Trung bình mAP từ IoU=0.5 đến 0.95
      - Precision / Recall theo từng class

    Args:
        weights_path: Đường dẫn tới file best.pt.
        cfg: Object cấu hình PipelineConfig.
    """
    print('\n' + '=' * 60)
    print('📊 ĐÁNH GIÁ MÔ HÌNH TRÊN TẬP TEST')
    print('   (Dữ liệu mô hình chưa bao giờ thấy trong training)')
    print('=' * 60)

    # Kiểm tra file weights tồn tại
    if not os.path.exists(weights_path):
        print(f'⚠️ Không tìm thấy weights: {weights_path}')
        print('   → Hãy chạy Cell 2 (Training) trước.')
        return

    # Load mô hình tốt nhất vừa train xong
    best_model = YOLO(weights_path)
    print(f'✅ Đã load model từ: {weights_path}')

    # Chạy validation trên split='test'
    # Ultralytics sẽ đọc key 'test' trong data.yaml
    print('\n🔄 Đang chấm điểm trên tập test...')
    metrics = best_model.val(
        split='test',               # Dùng tập test, KHÔNG phải val
        imgsz=cfg.IMAGE_SIZE,        # Giữ cùng img size với training
        batch=cfg.BATCH_SIZE,
        device=cfg.DEVICE,
        verbose=True,
    )

    # ---- In kết quả Segmentation metrics ----
    print('\n' + '-' * 50)
    print('🏆 KẾT QUẢ ĐÁNH GIÁ (Segmentation Masks)')
    print('-' * 50)
    print(f'  mAP50          : {metrics.seg.map50:.4f}')
    print(f'  mAP50-95       : {metrics.seg.map:.4f}')

    # In Precision / Recall trung bình
    if hasattr(metrics.seg, 'mp') and hasattr(metrics.seg, 'mr'):
        print(f'  Precision (TB) : {metrics.seg.mp:.4f}')
        print(f'  Recall (TB)    : {metrics.seg.mr:.4f}')

    # In mAP50 theo từng class (nếu có)
    if hasattr(metrics.seg, 'maps') and metrics.seg.maps is not None:
        print('\n  Chi tiết mAP50-95 theo class:')
        for i, class_map in enumerate(metrics.seg.maps):
            class_name = cfg.CLASS_NAMES[i] if i < len(cfg.CLASS_NAMES) else f'Class {i}'
            print(f'    {class_name:.<30s} {class_map:.4f}')

    print('-' * 50)
    print('✅ Đánh giá hoàn tất!')


# ============================================================
# 3.2 — HÀM INFERENCE THỰC TẾ (PRACTICAL — KHÔNG CÓ NHÃN)
# ============================================================
def run_inference(
    weights_path: str,
    image_source,  # PIL Image, URL string, hoặc file path
    cfg: PipelineConfig,
) -> None:
    """
    Chạy Instance Segmentation trên ảnh THỰC TẾ (practical)
    và hiển thị kết quả trực quan.

    Khác với evaluate_on_test(): hàm này KHÔNG tính mAP vì ảnh
    thực tế không có ground-truth label.

    Hỗ trợ 3 loại input:
      - URL (str bắt đầu bằng 'http')
      - File path (str trỏ đến file trên đĩa)
      - PIL.Image object

    Args:
        weights_path: Đường dẫn tới file .pt (best.pt).
        image_source: URL, filepath, hoặc PIL.Image.
        cfg: Object cấu hình PipelineConfig.
    """
    print('\n🔬 INFERENCE TRÊN ẢNH THỰC TẾ (PRACTICAL)')
    print('-' * 40)

    # Kiểm tra file weights tồn tại
    if not os.path.exists(weights_path):
        print(f'⚠️ Không tìm thấy weights: {weights_path}')
        print('   → Hãy chạy Cell 2 (Training) trước.')
        return

    # Load model đã train xong
    best_model = YOLO(weights_path)
    print(f'✅ Đã load model từ: {weights_path}')

    # ---- Xử lý input ảnh ----
    if isinstance(image_source, str) and image_source.startswith('http'):
        # Tải ảnh từ URL
        print(f'🌐 Đang tải ảnh từ URL...')
        try:
            response = requests.get(image_source, timeout=30)
            response.raise_for_status()
            img_pil = Image.open(BytesIO(response.content)).convert('RGB')
        except Exception as e:
            print(f'❌ Lỗi khi tải ảnh: {e}')
            return
    elif isinstance(image_source, str):
        # Load từ file path
        if not os.path.exists(image_source):
            print(f'❌ File không tồn tại: {image_source}')
            return
        img_pil = Image.open(image_source).convert('RGB')
    elif isinstance(image_source, Image.Image):
        img_pil = image_source.convert('RGB')
    else:
        print(f'❌ Loại input không hỗ trợ: {type(image_source)}')
        return

    print(f'📐 Ảnh input: {img_pil.size[0]}×{img_pil.size[1]}')

    # ---- Chạy prediction ----
    print('🔄 Đang chạy Instance Segmentation...')
    preds = best_model.predict(
        source=img_pil,
        conf=cfg.CONF_THRESH,          # Ngưỡng confidence
        iou=cfg.IOU_THRESH,            # Ngưỡng IoU cho NMS
        agnostic_nms=cfg.AGNOSTIC_NMS, # Gộp NMS qua các class
        imgsz=cfg.INFER_IMG_SIZE,      # Kích thước inference
        save=False,                     # Không auto-save (ta tự xử lý)
        verbose=False,
    )

    result = preds[0]

    # ---- Thống kê kết quả ----
    num_detections = len(result.boxes) if result.boxes is not None else 0
    print(f'📊 Phát hiện {num_detections} vùng bệnh')

    if num_detections > 0 and result.boxes is not None:
        # In chi tiết từng detection
        for i, (cls_id, conf) in enumerate(
            zip(result.boxes.cls.cpu().numpy(), result.boxes.conf.cpu().numpy())
        ):
            class_name = cfg.CLASS_NAMES[int(cls_id)]
            print(f'  [{i+1}] {class_name}: {conf:.2%}')

    # ---- Hiển thị ảnh kết quả ----
    # YOLO .plot() trả về mảng BGR → cần convert sang RGB cho matplotlib
    plotted_bgr = result.plot(
        boxes=True, labels=True, conf=True,
        line_width=2, font_size=18,
    )
    img_rgb = cv2.cvtColor(plotted_bgr, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(14, 11))
    plt.imshow(img_rgb)
    plt.axis('off')
    plt.title(
        f'Practical Inference — {num_detections} vùng bệnh (conf ≥ {cfg.CONF_THRESH})',
        fontsize=14, pad=12,
    )
    plt.tight_layout()
    plt.show()

    # ---- Lưu ảnh kết quả ra đĩa ----
    timestamp = int(time.time())
    output_path = os.path.join(cfg.WORKING_DIR, f'practical_result_{timestamp}.jpg')
    cv2.imwrite(output_path, plotted_bgr)  # cv2.imwrite nhận BGR
    print(f'💾 Ảnh kết quả đã lưu: {output_path}')


# ============================================================
# 3.3 — HÀM NÉN XUẤT KẾT QUẢ
# ============================================================
def export_results(source_dir: str, output_name: str = 'Ket_Qua_Train_Lua') -> None:
    """
    Nén toàn bộ thư mục kết quả training thành file .zip
    để team download về máy cá nhân.

    Args:
        source_dir: Thư mục cần nén (thường là PROJECT_DIR).
        output_name: Tên file zip (không cần đuôi .zip).
    """
    print('\n📦 ĐÓNG GÓI KẾT QUẢ')
    print('-' * 40)

    if not os.path.exists(source_dir):
        print(f'❌ Thư mục không tồn tại: {source_dir}')
        return

    # Tính kích thước thư mục trước khi nén
    total_size = sum(
        f.stat().st_size
        for f in Path(source_dir).rglob('*')
        if f.is_file()
    )
    size_mb = total_size / (1024 * 1024)
    print(f'📊 Kích thước thư mục: {size_mb:.1f} MB')

    output_path = os.path.join(CFG.WORKING_DIR, output_name)
    shutil.make_archive(output_path, 'zip', source_dir)

    zip_size = os.path.getsize(f'{output_path}.zip') / (1024 * 1024)
    print(f'✅ Đã nén thành công: {output_path}.zip ({zip_size:.1f} MB)')
    print('   → Vào tab "Output" của Kaggle Notebook để tải về.')


# ============================================================
# 3.4 — CHẠY: ĐÁNH GIÁ TEST → INFERENCE THỰC TẾ → XUẤT KẾT QUẢ
# ============================================================

# ---- Bước A: Đánh giá trên tập TEST (dữ liệu có nhãn, chưa từng train) ----
evaluate_on_test(
    weights_path=CFG.best_weights_path,
    # weights_path='/kaggle/input/đường_dẫn_của_bạn/best.pt',  # Mở comment dòng này và đóng dòng trên nếu chỉ muốn load trọng số có sẵn để test
    cfg=CFG,
)

# ---- Bước B: Inference trên ảnh THỰC TẾ (practical — không có nhãn) ----
run_inference(
    weights_path=CFG.best_weights_path,
    # image_source=CFG.SAMPLE_PRACTICAL_URL,
    # image_source='/kaggle/input/datasets/phantunqucanh/pratical-set/bac_la_2.jpg',
    cfg=CFG,
)

# ---- Bước C: Nén toàn bộ kết quả training ----


---
## BƯỚC 4: NÉN VÀ TẢI KẾT QUẢ VỀ MÁY (DOWNLOAD ARTIFACTS)
> 📦 **Hành động của Team:** Vì phiên Kaggle là phiên chạy mượn và sẽ biến mất khi tắt. Hãy chạy các Cell dưới đây để ZIP thư mục output (bao gồm trọng số, đồ thị loss) và tạo link trực tiếp tải gọn nhẹ về máy.

In [ ]:
# Nén toàn bộ thư mục model output thành 1 file ZIP duy nhất
!zip -r Rice_Disease_Seg.zip /kaggle/working/Rice_Disease_Seg -q
print("✅ Nén hoàn tất! File Rice_Disease_Seg.zip đã sẵn sàng để tải về.")

from IPython.display import FileLink
# Tạo link download HTML để ấn thẳng trong notebook
FileLink(r'Rice_Disease_Seg.zip')
